# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [1]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path('/home/jovyan/work')
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'wikipedia_id-raw.csv'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'wikipedia_id-bersih.parquet'

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset tidak ditemukan: {DATA_PATH}. '
        'Pastikan Milestone 1 sudah menempatkan data di data/raw/.'
    )

# LazyFrame digunakan agar seluruh CSV tidak langsung dimuat ke memori.
df = pl.scan_csv(DATA_PATH, null_values='', schema_overrides={
    'id': pl.Int64, 'parent_id': pl.Float64, 'timestamp': pl.String,
    'hash': pl.String, 'title': pl.String, 'text': pl.String,
})
jumlah_baris = df.select(pl.len()).collect(engine='streaming').item()
print(f'Ukuran awal: {jumlah_baris:,} baris x {len(df.collect_schema())} kolom')
df.head().collect(engine='streaming')

Ukuran awal: 1,550,872 baris x 6 kolom


id,parent_id,timestamp,hash,title,text
i64,f64,str,str,str,str
1,1.9472643e7,"""2022-04-24T00:35:51Z""","""oo2rvi3f9p4u9czu3opfvbsl0jp9ao…","""Asam deoksiribonukleat""","""[[Berkas:DNA Structure+Key+Lab…"
2,1.7982206e7,"""2021-02-15T10:17:21Z""","""49bjs0f6ys3j5ymkyg8c41ino347yk…","""Asam Deoksiribosanukleat""","""#ALIH [[Asam deoksiribonukleat…"
3,1.9395623e7,"""2022-05-05T11:26:32Z""","""lq3uz0weougma29prpmkm1l5at48s4…","""Anwar Sadat""","""{{Infobox officeholder | nativ…"
4,1.4292725e7,"""2021-07-05T02:41:20Z""","""3h1z248a5ic1u3odrse27wg0o0sllj…","""Azhar Mansor""","""[[Berkas:Azhar_Mansor_Jalur_Ge…"
5,2.1368465e7,"""2022-09-26T04:25:33Z""","""ololfquhgvu4wzszfu1yaqo3amb8xn…","""Arkeologi""","""{{Terjemah|Inggris}} {{Ilmu}} …"


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [2]:
connection = duckdb.connect()

profil = connection.execute(
    """
    SUMMARIZE SELECT * FROM read_csv(
        ?, header = true, delim = ',', quote = '"', escape = '"', nullstr = '',
        columns = {
            'id': 'BIGINT', 'parent_id': 'DOUBLE', 'timestamp': 'VARCHAR',
            'hash': 'VARCHAR', 'title': 'VARCHAR', 'text': 'VARCHAR'
        },
        max_line_size = 33554432
    )
    """,
    [str(DATA_PATH)],
).pl()
profil

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""id""","""BIGINT""","""1""","""3768441""",1985843,"""1827983.623075921""","""1124996.9688425125""","""772240""","""1912170""","""2798093""",1550872,0.00
"""parent_id""","""DOUBLE""","""2316.0""","""22030575.0""",1121672,"""15271049.586341834""","""5669601.806865775""","""12312131.909609893""","""16999028.566526514""","""20081289.523384236""",1550872,38.82
"""timestamp""","""VARCHAR""","""2004-04-19T13:00:14Z""","""2022-11-20T10:49:11Z""",1339674,null,null,null,null,null,1550872,0.00
"""hash""","""VARCHAR""","""0000ymuyndr6l1q4hc4kfdhgnrcpbi…","""twj48pdguc12bpfm814uum83mdnrl0…",1355277,null,null,null,null,null,1550872,0.00
"""title""","""VARCHAR""","""!""","""🪙""",1551409,null,null,null,null,null,1550872,0.00
"""text""","""VARCHAR""","""! colspan=""2"" style=""color:#ff…","""🚉 <noinclude>[[Kategori:Jalan …",1527130,null,null,null,null,null,1550872,0.06


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [3]:
null_counts = df.select(pl.all().null_count()).collect(engine='streaming')
null_summary = (
    null_counts
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null')
    )
    .filter(pl.col('jumlah_null') > 0)
    .sort('jumlah_null', descending=True)
)
null_summary

kolom,jumlah_null,persen_null
str,u32,f64
"""parent_id""",602033,38.82
"""text""",993,0.06
"""title""",1,0.0


## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [4]:
df_missing_handled = df.with_columns([
    pl.col('title').fill_null('tidak diketahui'),
    pl.col('text').fill_null(''),
])

null_sesudah = df_missing_handled.select(pl.all().null_count()).collect(engine='streaming')
print(f'Null sebelum: {null_counts.sum_horizontal().sum():,}')
print(f'Null sesudah: {null_sesudah.sum_horizontal().sum():,}')

Null sebelum: 603,027
Null sesudah: 602,033


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [5]:
KOLOM_NUMERIK = 'panjang_teks'
df_missing_handled = df_missing_handled.with_columns(
    pl.col('text').str.len_chars().alias(KOLOM_NUMERIK)
)
kuartil = (
    df_missing_handled.filter(pl.col(KOLOM_NUMERIK) > 0)
    .select([
        pl.col(KOLOM_NUMERIK).quantile(0.25, interpolation='linear').alias('q1'),
        pl.col(KOLOM_NUMERIK).quantile(0.75, interpolation='linear').alias('q3'),
    ])
    .collect(engine='streaming')
)
q1, q3 = kuartil.row(0)
iqr = q3 - q1
batas_bawah = max(0.0, q1 - 1.5 * iqr)
batas_atas = q3 + 1.5 * iqr

print(f'Q1={q1}, Q3={q3}, IQR={iqr}')
print(f'Batas bawah: {batas_bawah}, Batas atas: {batas_atas}')

# Winsorization: clip panjang teks, bukan memotong isi artikel.
df_outlier_handled = df_missing_handled.with_columns(
    pl.col(KOLOM_NUMERIK).cast(pl.Float64).clip(lower_bound=batas_bawah, upper_bound=batas_atas)
)

Q1=36.0, Q3=1424.0, IQR=1388.0
Batas bawah: 0.0, Batas atas: 3506.0


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [6]:
KOLOM_KUNCI = 'id'

ringkasan_duplikat = df_outlier_handled.select([
    pl.len().alias('total_baris'),
    pl.col(KOLOM_KUNCI).n_unique().alias('baris_unik'),
]).collect(engine='streaming')
total_baris, baris_unik = ringkasan_duplikat.row(0)
print(f'Total baris: {total_baris:,}')
print(f'Baris unik berdasarkan {KOLOM_KUNCI}: {baris_unik:,}')
print(f'Duplikat: {total_baris - baris_unik:,}')

df_dedup = df_outlier_handled.unique(
    subset=[KOLOM_KUNCI],
    keep='first',
    maintain_order=False,
)

Total baris: 1,550,872
Baris unik berdasarkan id: 1,550,872
Duplikat: 0


## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [7]:
connection.register('dataset_bersih', df_dedup.select([
    'id', 'title', KOLOM_NUMERIK,
    pl.col('text').is_null().alias('text_null'),
]).collect(engine='streaming'))

validasi = connection.execute(f"""
    SELECT
        COUNT(*) AS total_baris,
        SUM(CASE WHEN {KOLOM_NUMERIK} IS NULL THEN 1 ELSE 0 END) AS pelanggaran_null,
        SUM(CASE WHEN {KOLOM_NUMERIK} < {batas_bawah} OR {KOLOM_NUMERIK} > {batas_atas}
            THEN 1 ELSE 0 END) AS pelanggaran_batas,
        SUM(CASE WHEN title IS NULL OR text_null THEN 1 ELSE 0 END) AS judul_teks_null,
        SUM(CASE WHEN id IS NULL OR id <= 0 THEN 1 ELSE 0 END) AS id_tidak_valid,
        COUNT(*) - COUNT(DISTINCT id) AS duplikat_id
    FROM dataset_bersih
""").pl()
assert all(nilai == 0 for nilai in validasi.drop('total_baris').row(0)), 'Validasi data belum lulus.'
validasi

total_baris,pelanggaran_null,pelanggaran_batas,judul_teks_null,id_tidak_valid,duplikat_id
i64,"decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]",i64
1550872,0,0,0,0,0


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [8]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.sink_parquet(OUTPUT_PATH)

print(f'Baris sebelum cleaning: {jumlah_baris:,}')
print(f"Baris sesudah cleaning: {validasi.item(0, 'total_baris'):,}")
print(f'Data bersih tersimpan di: {OUTPUT_PATH}')
connection.close()

Baris sebelum cleaning: 1,550,872
Baris sesudah cleaning: 1,550,872
Data bersih tersimpan di: /home/jovyan/work/data/processed/wikipedia_id-bersih.parquet


## Catatan Milestone 2

Dokumentasikan di sini: dimensi kualitas data yang paling bermasalah, strategi yang dipilih untuk missing values dan outlier, jumlah baris yang berubah, serta aturan validasi yang diterapkan.

- Masalah utama adalah missing values: parent_id 602.033, title 1, dan text 993 record berdasarkan pemeriksaan CSV pada Milestone 1.

- Judul null diisi tidak diketahui, teks null diisi string kosong, dan null parent_id dipertahankan karena merupakan ID referensi. Total null pada enam kolom asli menjadi 602.033.

- Outlier ditangani dengan IQR dan clipping pada panjang_teks, dengan kuartil dari teks non-kosong. Isi artikel tetap utuh. Panjang asli dapat dihitung kembali dari text; jangan gunakan nilai hasil clipping sebagai panjang sebenarnya.

- Deduplikasi memakai id, dengan mempertahankan kemunculan pertama. Berdasarkan pemeriksaan CSV, tidak ada duplikat ID sehingga jumlah record tetap 1.550.872. Urutan baris keluaran tidak dijamin.

- Validasi mencakup null dan batas panjang teks, judul/teks non-null, serta ID positif dan unik. Hasil disimpan ke data/processed/wikipedia_id-bersih.parquet, dengan enam kolom asli dan satu kolom panjang_teks.